# Clade Constraints for Coalescent Simulations

This notebook demonstrates how to use clade constraints to enforce that specific groups of samples must form exclusive clades - meaning they must fully coalesce with each other before any can coalesce with samples outside the group.

In [ ]:
import numpy as np
from tree_constraint import CladeConstraint, CladeConstraintSet
from coalescent_constraints import Lineage

## 1. Creating Clade Constraints

A clade constraint is specified as a numpy array where 1s mark the samples that must form an exclusive clade. There are two ways to create a constraint:

In [ ]:
# Method 1: From a numpy array
mask = np.array([1, 1, 1, 0, 0, 0])
constraint1 = CladeConstraint(mask)
print(f"From array: {constraint1}")
print(f"Indices: {constraint1.indices}")
print(f"Clade size: {constraint1.clade_size()}")
print()

# Method 2: From a set of indices
constraint2 = CladeConstraint.from_indices({3, 4, 5}, n=6)
print(f"From indices: {constraint2}")
print(f"Indices: {constraint2.indices}")

## 2. Understanding Coalescence Rules

A clade constraint enforces that:
1. Samples in the clade can only coalesce with each other
2. Samples outside the clade can coalesce freely with each other
3. Clade and non-clade samples cannot mix until the clade is fully coalesced

In [ ]:
# Create a constraint: samples 0, 1, 2 must form a clade
constraint = CladeConstraint.from_indices({0, 1, 2}, n=6)
print(f"Constraint: {constraint}")
print()

# Create individual lineages
lineages = {Lineage.from_sample(i, 6) for i in range(6)}
lin = {i: Lineage.from_sample(i, 6) for i in range(6)}

print("Coalescence possibilities:")
print(f"  0 + 1 (both in clade):     {constraint.can_coalesce(lin[0], lin[1])}")
print(f"  0 + 2 (both in clade):     {constraint.can_coalesce(lin[0], lin[2])}")
print(f"  3 + 4 (both outside):      {constraint.can_coalesce(lin[3], lin[4])}")
print(f"  4 + 5 (both outside):      {constraint.can_coalesce(lin[4], lin[5])}")
print(f"  0 + 3 (clade + outside):   {constraint.can_coalesce(lin[0], lin[3])}")
print(f"  2 + 5 (clade + outside):   {constraint.can_coalesce(lin[2], lin[5])}")

## 3. Tracking Constraint Satisfaction

A constraint is satisfied when all samples in the clade have coalesced into a single lineage.

In [ ]:
constraint = CladeConstraint.from_indices({0, 1}, n=6)
print(f"Constraint: {constraint}")

# Initial state: all separate
lineages = {Lineage.from_sample(i, 6) for i in range(6)}
print(f"\nAll separate - satisfied: {constraint.is_satisfied_by(lineages)}")

# After merging 0 and 1
merged_01 = Lineage.coalesce(Lineage.from_sample(0, 6), Lineage.from_sample(1, 6))
lineages = {merged_01} | {Lineage.from_sample(i, 6) for i in range(2, 6)}
print(f"After merging 0+1 - satisfied: {constraint.is_satisfied_by(lineages)}")

## 4. Managing Multiple Constraints

`CladeConstraintSet` manages multiple constraints and ensures no sample appears in more than one active constraint.

In [ ]:
cs = CladeConstraintSet()

# Add non-overlapping constraints
cs.add(CladeConstraint.from_indices({0, 1}, n=6))
cs.add(CladeConstraint.from_indices({3, 4, 5}, n=6))

print(f"Constraint set: {cs}")
print(f"Number of constraints: {len(cs)}")
print("\nConstraints:")
for c in cs:
    print(f"  {c}")

### Overlap Validation

Attempting to add a constraint with overlapping samples raises an error:

In [ ]:
cs = CladeConstraintSet()
cs.add(CladeConstraint.from_indices({0, 1, 2}, n=6))

try:
    # This will fail - sample 2 is already in the first constraint
    cs.add(CladeConstraint.from_indices({2, 3, 4}, n=6))
except ValueError as e:
    print(f"Error (expected): {e}")

## 5. Coalescence with Multiple Constraints

When checking if two lineages can coalesce, all constraints must allow it:

In [ ]:
cs = CladeConstraintSet()
cs.add(CladeConstraint.from_indices({0, 1}, n=6))  # Clade A
cs.add(CladeConstraint.from_indices({2, 3}, n=6))  # Clade B

lin = {i: Lineage.from_sample(i, 6) for i in range(6)}

print("With constraints {0,1} and {2,3}:")
print(f"  0 + 1 (same clade):        {cs.can_coalesce(lin[0], lin[1])}")
print(f"  2 + 3 (same clade):        {cs.can_coalesce(lin[2], lin[3])}")
print(f"  4 + 5 (no clade):          {cs.can_coalesce(lin[4], lin[5])}")
print(f"  0 + 2 (different clades):  {cs.can_coalesce(lin[0], lin[2])}")
print(f"  0 + 4 (clade + free):      {cs.can_coalesce(lin[0], lin[4])}")

## 6. Auto-updating After Coalescence

The constraint set can automatically remove satisfied constraints:

In [ ]:
cs = CladeConstraintSet()
cs.add(CladeConstraint.from_indices({0, 1}, n=6))
cs.add(CladeConstraint.from_indices({2, 3}, n=6))
print(f"Initial: {len(cs)} constraints")

# Merge samples 0 and 1 (satisfies first constraint)
merged_01 = Lineage.coalesce(Lineage.from_sample(0, 6), Lineage.from_sample(1, 6))
lineages = {merged_01} | {Lineage.from_sample(i, 6) for i in range(2, 6)}

cs.update_after_coalescence(lineages)
print(f"After merging 0+1: {len(cs)} constraints")

# Merge samples 2 and 3 (satisfies second constraint)
merged_23 = Lineage.coalesce(Lineage.from_sample(2, 6), Lineage.from_sample(3, 6))
lineages = {merged_01, merged_23, Lineage.from_sample(4, 6), Lineage.from_sample(5, 6)}

cs.update_after_coalescence(lineages)
print(f"After merging 2+3: {len(cs)} constraints")

# Now all lineages can coalesce freely
print(f"\nCan merged_01 + merged_23 coalesce? {cs.can_coalesce(merged_01, merged_23)}")

## 7. Efficient Pair Finding

The constraint set provides efficient methods to find compatible pairs without O(n²) pairwise checks:

In [ ]:
cs = CladeConstraintSet()
cs.add(CladeConstraint.from_indices({0, 1, 2}, n=8))
cs.add(CladeConstraint.from_indices({3, 4}, n=8))

lineages = {Lineage.from_sample(i, 8) for i in range(8)}

# Efficient grouping - lineages in same group can coalesce
print("Compatible groups:")
groups = cs.get_compatible_groups(lineages)
for i, group in enumerate(groups):
    print(f"  Group {i}: {[str(l) for l in group]}")

print("\nCompatible pairs:")
pairs = cs.get_compatible_pairs(lineages)
for a, b in pairs:
    print(f"  {a} + {b}")

## 8. Complete Simulation Example

Here's a complete example simulating coalescence with clade constraints, using efficient pair finding:

In [ ]:
import random

def simulate_with_clade_constraints(n_samples, clade_indices_list, seed=42):
    """
    Simulate coalescence with clade constraints.

    Args:
        n_samples: Total number of samples
        clade_indices_list: List of sets, each defining a clade
        seed: Random seed
    """
    random.seed(seed)

    # Set up constraints
    cs = CladeConstraintSet()
    for indices in clade_indices_list:
        cs.add(CladeConstraint.from_indices(indices, n_samples))

    # Initialize lineages
    lineages = {Lineage.from_sample(i, n_samples) for i in range(n_samples)}

    print(f"Samples: {n_samples}")
    print(f"Clade constraints: {[set(c.indices) for c in cs]}")
    print()

    step = 0
    while len(lineages) > 1:
        # Use efficient pair finding
        compatible = cs.get_compatible_pairs(lineages)

        if not compatible:
            print(f"Step {step}: No compatible pairs - deadlock!")
            break

        # Randomly choose a pair
        lin_a, lin_b = random.choice(compatible)
        merged = Lineage.coalesce(lin_a, lin_b)

        # Update lineages
        lineages.remove(lin_a)
        lineages.remove(lin_b)
        lineages.add(merged)

        # Update constraints
        cs.update_after_coalescence(lineages)

        print(f"Step {step}: {lin_a} + {lin_b} -> {merged}")
        print(f"         Active constraints: {len(cs)}")
        step += 1

    print(f"\nCoalescence complete in {step} steps")
    return lineages

In [ ]:
# Example 1: Two clades
simulate_with_clade_constraints(
    n_samples=6,
    clade_indices_list=[{0, 1, 2}, {3, 4}]
)

In [ ]:
# Example 2: Three small clades
simulate_with_clade_constraints(
    n_samples=8,
    clade_indices_list=[{0, 1}, {2, 3}, {4, 5}]
)

## 9. Comparison: Bipartition vs Clade Constraints

| Feature | Bipartition | CladeConstraint |
|---------|-------------|------------------|
| Specification | Divides ALL samples into A \| B | Marks a subset as a clade |
| Unconstrained samples | None - all samples assigned | Samples with 0 are free |
| Coalescence rule | Same partition only | Clade members only, or free samples only |
| Typical use | Tree topology constraints | Enforcing specific clades |